READING DATASET

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix,
                             roc_curve, auc, roc_auc_score)
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import label_binarize

In [ ]:

!pip install zenodo-get

!zenodo_get 17761467


In [ ]:

# ---- Read CSV ----
csv_path = "SDN-Net_dataset.csv"
if not os.path.exists(csv_path):
    raise FileNotFoundError(f"CSV file not found at {csv_path}")

df = pd.read_csv(csv_path, skipinitialspace=True, low_memory=False)
print("Initial shape:", df.shape)
print("Columns:", df.columns.tolist())


In [ ]:
print(df['Class'].value_counts())

In [ ]:
# Assuming 'df' is your DataFrame and 'Class' is the column you want to drop
class_data = df['Class'].copy()


In [ ]:
attacks_types = {
'NORMAL': 'normal',
'DOS': 'attack',
'DDOS': 'attack',
'WEB ATTACK � BRUTE FORCE': 'attack',
'WEB ATTACK � XSS': 'attack',
'WEB ATTACK � SQL INJECTION': 'attack',
'WEB-ATTACK':'attack',
'U2R': 'attack',
'PROBE': 'attack',
'BFA': 'attack',
'BOTNET': 'attack',
}


In [ ]:
# Correcting the dictionary key access and fixing the syntax error
df['Attack Type'] = df['Class'].apply(lambda r: attacks_types[r])

# Display the first few rows of the dataframe to check the changes
df.head()


In [ ]:
df.drop('Unnamed: 0', axis=1, inplace=True)
# Display the first few rows of the dataframe to check the changes

df.head()

In [ ]:
df.shape

In [ ]:
df['Attack Type'].value_counts()

In [ ]:
df['Attack Type'].value_counts()

In [ ]:
df.dtypes

DATA PREPROCESSING

In [ ]:
df.isnull().sum()

In [ ]:
#Finding categorical features
num_cols = df._get_numeric_data().columns

cate_cols = list(set(df.columns)-set(num_cols))
cate_cols.remove('Class')
cate_cols.remove('Attack Type')

cate_cols

CATEGORICAL FEATURES DISTRIBUTION

In [ ]:
#Visualization
def bar_graph(feature):
    df[feature].value_counts().plot(kind="bar")

TARGET FEATURE DISTRIBUTION

In [ ]:
bar_graph('Class')

Attack Type(The attack types grouped by attack, it's what we will predict)

In [ ]:
bar_graph('Attack Type')

In [ ]:
df.columns

In [ ]:
df = df.dropna(axis='columns')  # Correct usage to drop columns with NaN values
df = df.dropna(axis=1)  # Using axis=1 instead of 'columns' also works


MODELLING

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score

In [ ]:
df.drop('Class', axis=1, inplace=True)
print(df.shape)

In [ ]:
print(df.shape)
print(df.head)

In [ ]:
# Identify columns with infinite values
cols_with_infinite = df.columns[df.isin([np.inf, -np.inf]).any()].tolist()

# Check for missing values in these columns
missing_data = df[cols_with_infinite].isnull().sum()

# Handle missing data (replace infinite values with the mean)
for col in cols_with_infinite:
    df[col].replace([np.inf, -np.inf], np.nan, inplace=True)
    df[col].fillna(df[col].mean(), inplace=True)

# Recheck for infinite values
cols_with_infinite = df.columns[df.isin([np.inf, -np.inf]).any()].tolist()

# Now cols_with_infinite should be an empty list, indicating there are no more infinite values
if not cols_with_infinite:
    print("No more infinite values in the df dataset.")


In [ ]:

print(df.shape)
print(df.columns)

In [ ]:
# Data Preparation
X = df.drop('Attack Type', axis=1)
Y = df['Attack Type']

# Encode the target variable using LabelEncoder
label_encoder = LabelEncoder()
Y = label_encoder.fit_transform(Y)

X_train, X_test, Y_train, y_test = train_test_split(X, Y, test_size=0.33, random_state=42)

# Replace infinite values with 0
X_train.replace([np.inf, -np.inf], 0, inplace=True)
X_test.replace([np.inf, -np.inf], 0, inplace=True)

# Apply Standard Scaling to features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [ ]:
print(X_train.shape, X_test.shape)
print(Y_train.shape, y_test.shape)

In [ ]:
from sklearn.model_selection import learning_curve, cross_val_score
from sklearn.neighbors import KNeighborsClassifier  # Import K-Nearest Neighbors Classifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, roc_curve, roc_auc_score

# Assuming you have your data loaded as X (features) and y (target)

# Define the K-Nearest Neighbors model
model = KNeighborsClassifier()

# Learning Curve
train_sizes, train_scores, valid_scores = learning_curve(model, X, Y, train_sizes=np.linspace(0.1, 1.0, 10), cv=5)

plt.figure()
plt.plot(train_sizes, np.mean(train_scores, axis=1), 'o-', label='Training score')
plt.plot(train_sizes, np.mean(valid_scores, axis=1), 'o-', label='Cross-validation score')
plt.xlabel('Training examples')
plt.ylabel('Score')
plt.title('Learning Curve (K-Nearest Neighbors)')
plt.legend()
plt.grid(True)
plt.show()

# Optionally, you can print the cross-validation scores
cv_scores = cross_val_score(model, X, Y, cv=5)
print("Cross-Validation Scores:", cv_scores)
print("Average Cross-Validation Score:", np.mean(cv_scores))

# Train the classifier
history = model.fit(X_train, Y_train)

# Predict on the test set
y_pred = model.predict(X_test)

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

# Calculate confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Print evaluation metrics
print(f"Model: {type(model).__name__}")
print("Testing accuracy: {:.2%}".format(accuracy))
print("Precision: {:.2%}".format(precision))
print("Recall: {:.2%}".format(recall))
print("F1-score: {:.2%}".format(f1))
print("Confusion Matrix:")
print(cm)
print("\n")

# Print the classification report
class_report = classification_report(y_test, y_pred, target_names=df['Attack Type'].unique())
print("Model : K-Nearest Neighbors")
print(class_report)
print("\n")

# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()
